# 02 CNN Advanced Architectures

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- Build a small **residual-style** block (skip connection) and compare with a plain stack of conv layers
- See how residual connections help training (optional: loss curve comparison)
- Understand why we use ResNet-style architectures instead of very deep plain CNNs

---

## 🌍 Real life

**Where is this used?** ResNet, VGG, Inception are used in **image classification**, **object detection**, and **segmentation** in industry and research.

**In this notebook we use** a **residual block** (conv + skip connection) so the network can learn **residuals** instead of full mappings. We use **skip connections** (instead of a plain deep stack of conv layers) **because** they help gradients flow and allow training **deeper** networks without vanishing gradients.

**📌 Covers slide(s):** **16** — CNN Architectures (LeNet, AlexNet). *Do this notebook after that slide.*

---

**Before starting:** Run the imports cell below.

## Theory (short)

- **ResNet (residual network):** Each block computes F(x) and outputs **x + F(x)** (skip connection). The network learns **residuals** (what to add) instead of the full mapping.
- **Why residuals?** Very deep plain networks can suffer from vanishing gradients; skip connections give a direct path for gradients and make optimization easier.
- **VGG / Inception:** VGG = many small 3×3 convs; Inception = multiple filter sizes in parallel. We focus on the residual idea here.
- **We use a residual block** instead of only conv layers so we can go deeper without losing gradient flow.

## 📥 Inputs & 📤 Outputs

**Inputs:** TensorFlow/Keras, NumPy, MNIST (small subset). We build a tiny model with one residual block.

**Dataset:** Real — MNIST (small subset).

**Outputs:** Model summary showing residual block structure; optional short training to show it runs.

## Step 1: Imports

In [1]:
import numpy as np

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print("⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.")
        raise RuntimeError("Fix: pip install --upgrade charset-normalizer requests, then restart kernel.") from e
    HAS_TF = False

print("TensorFlow:", "yes" if HAS_TF else "no")

TensorFlow: yes


## Step 2: Define a residual block (we use skip connection so gradients flow and we can train deeper nets)

In [2]:
if HAS_TF:
    def residual_block(x, filters):
        shortcut = x
        x = keras.layers.Conv2D(filters, (3, 3), padding="same", activation="relu")(x)
        x = keras.layers.Conv2D(filters, (3, 3), padding="same")(x)
        x = keras.layers.Add()([x, shortcut])
        x = keras.layers.Activation("relu")(x)
        return x

    inp = keras.Input(shape=(28, 28, 1))
    x = keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inp)
    x = residual_block(x, 32)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inp, x)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.summary()
    print("\nResidual block: conv → conv → Add(shortcut) → ReLU. Skip connection helps gradient flow.")

Model: "model"


__________________________________________________________________________________________________


 Layer (type)                Output Shape                 Param #   Connected to                  


 input_1 (InputLayer)        [(None, 28, 28, 1)]          0         []                            


 conv2d (Conv2D)             (None, 28, 28, 32)           320       ['input_1[0][0]']             


...

Total params: 19146 (74.79 KB)


Trainable params: 19146 (74.79 KB)


Non-trainable params: 0 (0.00 Byte)


__________________________________________________________________________________________________



Residual block: conv → conv → Add(shortcut) → ReLU. Skip connection helps gradient flow.


## Step 3: Train on MNIST subset (2 epochs to verify it runs)

In [3]:
if HAS_TF:
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
    x_train = x_train.astype(np.float32) / 255.0
    x_test = x_test.astype(np.float32) / 255.0
    x_train = x_train[..., np.newaxis][:5000]
    y_train = y_train[:5000]
    x_test = x_test[..., np.newaxis]
    history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=2, batch_size=128, verbose=1)
    print("Final val accuracy: %.4f" % history.history["val_accuracy"][-1])

Epoch 1/2


 1/40 [..............................] - ETA: 5s - loss: 2.3034 - accuracy: 0.0625

 3/40 [=>............................] - ETA: 1s - loss: 2.3049 - accuracy: 0.0833

 5/40 [==>...........................] - ETA: 1s - loss: 2.3033 - accuracy: 0.0859

 7/40 [====>.........................] - ETA: 1s - loss: 2.3028 - accuracy: 0.1172

...

35/40 [=========================>....] - ETA: 0s - loss: 2.2146 - accuracy: 0.1964

37/40 [==========================>...] - ETA: 0s - loss: 2.2077 - accuracy: 0.2025

39/40 [============================>.] - ETA: 0s - loss: 2.1993 - accuracy: 0.2087

40/40 [==============================] - 2s 50ms/step - loss: 2.1990 - accuracy: 0.2088 - val_loss: 2.0382 - val_accuracy: 0.2658


Final val accuracy: 0.2658


## 🧩 Mini-exercise

**Try it:** In a new cell, build a second residual block (same pattern: Conv → Conv → Add(shortcut)) and add it to the model, then train for 1 epoch. Compare the number of parameters with the one-block model.

---

## ✅ Summary

**What you did:** Built a small model with a residual block (skip connection) and trained it on MNIST. Saw how Add(shortcut) is used.

**In real life you'd also:** Use full ResNet/VGG from Keras Applications, more blocks, and ImageNet pre-training.

**The main idea:** Residual connections (x + F(x)) let gradients flow and allow training deeper CNNs; ResNet-style architectures are standard for image tasks.

**Next:** `05_transfer_learning_cnns` uses pre-trained models; `06_pretrained_cnn_architectures` explores ResNet/VGG/Inception.